    # P1 修复后结果复盘

本 notebook 只读取 `outputs/p1_main_20260324/all_models` 下的结果文件，用于支撑报告收口。

结论使用边界：
- 数据窗口：`2020-01-02` 到 `2022-12-30`
- 本次已修复：`target` 尾行误标 `0` 的 P1 问题
- 仍未修复：CNN 验证集 `random_split` 的时间泄漏，因此 CNN 只能作为探索结果，不作为主结论
- 传统模型 `Logistic Regression / XGBoost` 更适合作为课程主线对比


In [1]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

OUTPUT_DIR = ROOT / 'outputs' / 'p1_main_20260324' / 'all_models'
COMPARISON_PATH = OUTPUT_DIR / 'model_comparison.csv'
CONCLUSIONS_PATH = OUTPUT_DIR / 'conclusions.json'

comparison_df = pd.read_csv(COMPARISON_PATH)
with open(CONCLUSIONS_PATH, 'r') as f:
    conclusions = json.load(f)

comparison_df


,symbol,model,accuracy,precision,recall,f1,roc_auc
0,0700.HK,logistic_regression,0.455172,0.449153,0.791045,0.572973,0.472445
1,0700.HK,xgboost,0.489655,0.472000,0.880597,0.614583,0.485075
2,0700.HK,cnn_specific,0.457746,0.457746,1.000000,0.628019,0.500000
3,0700.HK,cnn_universal,0.450000,0.450000,1.000000,0.620690,0.500000
4,BABA,logistic_regression,0.520270,0.000000,0.000000,0.000000,0.506311
5,BABA,xgboost,0.479730,0.440000,0.309859,0.363636,0.505396
6,BABA,cnn_specific,0.517241,0.000000,0.000000,0.000000,0.500000
7,BABA,cnn_universal,0.524476,0.000000,0.000000,0.000000,0.512549
8,BIDU,logistic_regression,0.486486,0.468085,0.628571,0.536585,0.482967
9,BIDU,xgboost,0.500000,0.474359,0.528571,0.500000,0.522161


## 1. 汇总表

先看单股-单模型层面的原始比较结果。报告写作时不要只看 `accuracy`，要同时看 `precision / recall / f1 / roc_auc`。


In [2]:
comparison_df.sort_values(['roc_auc', 'accuracy'], ascending=False).reset_index(drop=True)


,symbol,model,accuracy,precision,recall,f1,roc_auc
0,NVDA,xgboost,0.540541,0.525773,0.698630,0.600000,0.543562
1,GOOGL,xgboost,0.472973,0.445378,0.815385,0.576087,0.524560
2,BIDU,xgboost,0.500000,0.474359,0.528571,0.500000,0.522161
3,BABA,cnn_universal,0.524476,0.000000,0.000000,0.000000,0.512549
4,GOOGL,cnn_specific,0.572414,1.000000,0.015873,0.031250,0.507937
5,BABA,logistic_regression,0.520270,0.000000,0.000000,0.000000,0.506311
6,BABA,xgboost,0.479730,0.440000,0.309859,0.363636,0.505396
7,MSFT,cnn_specific,0.551724,0.000000,0.000000,0.000000,0.500000
8,BIDU,cnn_specific,0.531034,0.000000,0.000000,0.000000,0.500000
9,BABA,cnn_specific,0.517241,0.000000,0.000000,0.000000,0.500000


## 2. 模型平均表现

这个表最适合放进报告正文，用来说明不同模型家族的平均效果和稳定程度。


In [3]:
model_avg = (
    comparison_df.groupby('model')[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
    .mean()
    .sort_values('roc_auc', ascending=False)
    .round(4)
)
model_avg


,accuracy,precision,recall,f1,roc_auc
model,,,,,
xgboost,0.4893,0.4627,0.6284,0.5208,0.5111
cnn_specific,0.5211,0.3257,0.3360,0.2205,0.5013
cnn_universal,0.4666,0.3792,0.8333,0.5210,0.4959
logistic_regression,0.4745,0.3778,0.6207,0.4686,0.4857


## 3. 退化预测检查

如果某个模型经常出现 `precision=0`、`recall=0` 或 `recall=1`，说明它可能在某些股票上退化成“几乎总是预测同一类”。
这类结果可以写进 limitation，而不是包装成性能优势。


In [4]:
degenerate = comparison_df[
    (comparison_df['precision'] == 0)
    | (comparison_df['recall'] == 0)
    | (comparison_df['recall'] == 1)
].sort_values(['model', 'symbol']).reset_index(drop=True)
degenerate


,symbol,model,accuracy,precision,recall,f1,roc_auc
0,0700.HK,cnn_specific,0.457746,0.457746,1.0,0.628019,0.500000
1,BABA,cnn_specific,0.517241,0.000000,0.0,0.000000,0.500000
2,BIDU,cnn_specific,0.531034,0.000000,0.0,0.000000,0.500000
3,MSFT,cnn_specific,0.551724,0.000000,0.0,0.000000,0.500000
4,NVDA,cnn_specific,0.496552,0.496552,1.0,0.663594,0.500000
5,0700.HK,cnn_universal,0.450000,0.450000,1.0,0.620690,0.500000
6,BABA,cnn_universal,0.524476,0.000000,0.0,0.000000,0.512549
7,BIDU,cnn_universal,0.461538,0.461538,1.0,0.631579,0.500000
8,GOOGL,cnn_universal,0.426573,0.426573,1.0,0.598039,0.500000
9,MSFT,cnn_universal,0.447552,0.447552,1.0,0.618357,0.500000


## 4. ROC-AUC 主图

建议把 ROC-AUC 作为主排名指标，因为分类阈值下的 accuracy 很容易被类别偏置和单边预测误导。


In [5]:
ax = model_avg['roc_auc'].sort_values(ascending=False).plot(kind='bar', color=['#2a9d8f', '#e9c46a', '#f4a261', '#e76f51'])
ax.set_title('Average ROC-AUC by Model')
ax.set_xlabel('Model')
ax.set_ylabel('ROC-AUC')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


/var/folders/qy/4ql898l12095lvc828b1h6gw0000gn/T/ipykernel_71271/3644467050.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. 单股 x 模型热力图

这个图最适合支撑“不同股票对模型敏感度不同，因此必须做横向对比”的叙事。


In [6]:
pivot_roc = comparison_df.pivot(index='symbol', columns='model', values='roc_auc')
plt.figure(figsize=(9, 4.5))
sns.heatmap(pivot_roc, annot=True, fmt='.3f', cmap='YlOrBr', vmin=0.45, vmax=0.55)
plt.title('ROC-AUC Heatmap by Symbol and Model')
plt.tight_layout()
plt.show()


/var/folders/qy/4ql898l12095lvc828b1h6gw0000gn/T/ipykernel_71271/597081583.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Precision / Recall 对照

这个图用来解释为什么不能只看 accuracy。CNN 在若干股票上 recall 很高，但 precision 很低，说明它更像是单边预测，而不是有效识别涨跌。


In [7]:
plot_df = comparison_df.copy()
plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='recall', y='precision', hue='model', style='symbol', s=120)
plt.title('Precision vs Recall')
plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()


/var/folders/qy/4ql898l12095lvc828b1h6gw0000gn/T/ipykernel_71271/2963153841.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. 可直接用于报告的叙事骨架

### 技术叙事
- 我们首先修复了标签构造中的 P1 问题，避免每只股票最后一天被错误标记为下跌类。
- 在修复后数据上，对 `Logistic Regression`、`XGBoost`、`CNN (specific)`、`CNN (universal)` 做单股二分类比较。
- 结果显示各模型整体 ROC-AUC 都接近 `0.5`，说明仅靠价格与成交量衍生特征，预测下一日涨跌本身就很难。
- 其中 `XGBoost` 的平均 ROC-AUC 最高，更适合作为传统机器学习主线 baseline。
- CNN 虽然在部分股票上得到较高 recall，但伴随明显的单边预测迹象，因此更适合作为探索性扩展，而不是主结论。

### 业务叙事
- 该项目的现实价值不在于短期内做出可交易策略，而在于建立“数据获取 -> 特征工程 -> 模型对比 -> 结果解释”的完整量化实验链路。
- 对课程场景而言，重点贡献是验证了不同模型与特征路线在同一数据窗口下的表现边界，而不是声称达成稳定超额收益。
- 结果也说明，如果后续想继续提升，下一步更可能来自更高质量因子，而不是盲目堆更复杂的模型结构。

### 报告里建议重点展示的 4 张图
1. `Average ROC-AUC by Model`
2. `ROC-AUC Heatmap by Symbol and Model`
3. `Precision vs Recall`
4. `logistic_regression_vs_xgboost_roc_curves.png`（来自 `outputs/p1_main_20260324/all_models`）

### 不建议作为主结论的点
- 不要把 `recommended_model = cnn_specific` 这种自动规则输出直接写进报告。
- 不要把 CNN 的高 recall 直接解释成“更擅长预测上涨”。
- 不要把 `accuracy > 0.52` 包装成具备交易价值。


In [8]:
conclusions


{'best_performing': {'symbol': 'NVDA',
  'model': 'xgboost',
  'roc_auc': 0.5435616438356164},
 'model_stability': {'cnn_specific': 0.0032400657973322327,
  'cnn_universal': 0.016849339777238348,
  'logistic_regression': 0.016578997570276255,
  'xgboost': 0.02328644888342232},
 'stock_modelability': {'0700.HK': 0.4893800229621125,
  'BABA': 0.5060639057159355,
  'BIDU': 0.5012820512820513,
  'GOOGL': 0.49864770437059597,
  'MSFT': 0.49437995209139485,
  'NVDA': 0.5013714938030007},
 'model_type_performance': {'CNN': 0.49863308946105084,
  'Traditional': 0.4984086206139795},
 'pipeline_performance': {'Specific': 0.5013227513227513,
  'Universal': 0.49758688960910313},
 'recommendations': {'recommended_symbol': 'BABA',
  'recommended_model': 'cnn_specific',
  'recommended_model_type': 'CNN',
  'recommended_pipeline': 'Specific'}}